# 82장 · SQLite 영속성과 instr() 부분 문자열 검색

파일 기반 SQLite DB는 연결을 닫은 뒤에도 확정된 데이터를 보관합니다. instr()는 찾으려는 문자열이 처음 나타나는 위치를 돌려줍니다. %도 글자 그대로 검색할 수 있습니다.

할 일을 파일 DB에 저장하고 다시 연결해 같은 ID로 조회합니다. 81장의 API 필드를 유지하면서 제목에 포함된 %를 실제 문자로 검색하는 조건도 검사합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 파일 DB의 커밋과 재연결

수업 서버의 PostgreSQL 저장소는 트랜잭션을 커밋한 뒤 저장한 레코드를 돌려줍니다. 여기서는 임시 디렉터리의 SQLite 파일을 사용해 커밋과 새 연결의 차이를 직접 확인합니다. 실제 PostgreSQL 컨테이너는 시작하지 않습니다.


### 세 열로 저장하는 할 일 한 건

<p>todos 테이블은 id를 INTEGER PRIMARY KEY, title을 TEXT, completed를 INTEGER로 둡니다. 이 예제에서는 Python의 False가 SQLite의 정수 0으로 저장되므로 조회 결과의 마지막 값이 0입니다.</p>


### 셀 01 · SQLite 파일에 INSERT

저장한 행의 ID를 기록합니다. 커널이 살아 있는 동안만 임시 폴더를 유지합니다.

저장 결과가 사라진 것 같으면 파일 경로 → 테이블 존재 → 필터 없는 행 순서로 확인하세요. 새 파일에는 테이블이 없어 조회 오류가 나고, 빈 테이블에는 0행이 있습니다. 원래 파일에서 필터만 풀었을 때 ID 1이 보인다면 행은 남아 있습니다.


In [1]:
import sqlite3
from tempfile import TemporaryDirectory
from pathlib import Path
workspace = TemporaryDirectory()
db_path = Path(workspace.name) / "todos.db"
db = sqlite3.connect(db_path)
with db:
    db.execute("CREATE TABLE todos(id INTEGER PRIMARY KEY, title TEXT, completed INTEGER)")
    created = db.execute("INSERT INTO todos(title, completed) VALUES (?, ?)", ("복습 50%", False))
saved_id = created.lastrowid
db.close()
print(saved_id)

1


새 DB에서 첫 ID 1이 발급됐습니다. 연결은 닫았지만 파일은 지우지 않았습니다.


### SQL 자리표시자와 별도의 값

<p>제목과 완료 상태는 INSERT 문장에 직접 붙이지 않고 (?, ?)와 값 튜플로 전달합니다. 한 값만 전달하는 조회에서는 (saved_id,)처럼 쉼표가 있는 튜플을 사용합니다.</p>


### 셀 02 · 재연결 후 ID로 SELECT

두 번째 연결이 같은 파일을 가리키는지 살펴보세요.


In [2]:
db = sqlite3.connect(db_path)
print(db.execute("SELECT id,title,completed FROM todos WHERE id=?", (saved_id,)).fetchone())

(1, '복습 50%', 0)


다시 연 파일에서 (1, '복습 50%', 0)이 나옵니다. 같은 ID의 제목과 완료 값을 새 연결에서 읽었습니다. SQLite의 0은 이 테이블에서 미완료를 뜻하며, 웹 API로 돌려줄 때는 불리언으로 바꿉니다.


### 파일 경로를 고정한 재연결

<p>TemporaryDirectory 안에 todos.db를 만들고 정상 커밋한 뒤 연결을 닫습니다. 같은 경로로 sqlite3.connect를 다시 호출했을 때도 1번 행을 찾을 수 있으면 임시 폴더가 살아 있는 동안 파일에 값이 남았음을 확인한 것입니다.</p>


## instr()로 퍼센트 기호를 글자 그대로 검색

검색창에 %를 입력한 사용자는 그 기호가 제목에 들어간 할 일을 찾으려 합니다. PostgreSQL 제공 구현은 strpos로 부분 문자열을 검사합니다. SQLite의 비슷한 함수 instr는 찾은 위치를 1부터 세며, 없으면 0을 돌려줍니다.

<!-- lesson-visual practice-space -->
**퍼센트 기호가 들어간 행만 어떻게 찾을까요?**

원래 두 행에 같은 검색어를 넣습니다. SQLite의 instr()가 찾은 첫 위치를 표시합니다. 0은 해당 글자가 없다는 뜻입니다.

퍼센트 기호는 ‘복습 50%’의 여섯 번째 글자입니다. %를 찾으면 이 행만, ‘복습’을 찾으면 두 행이 모두 남습니다.

[웹 교재에서 직접 조작하기](https://langchain-lecture.vercel.app/chapter82/#practice-space)
<!-- /lesson-visual practice-space -->


### 완료 조건과 제목 검색을 함께 적용하기

<p>두 조건을 AND로 연결하면 제목에 복습이 있으면서 완료 상태인 행만 남습니다. 복습 50%도 제목 조건에는 맞지만 completed가 0이므로 완료 필터를 통과하지 못합니다.</p>


### 셀 03 · instr()로 부분 문자열 위치 찾기

비교할 일반 제목 한 건을 더 넣은 뒤 검색합니다.


In [3]:
with db:
    db.execute("INSERT INTO todos(title, completed) VALUES (?,?)", ("복습 마무리", True))
matches = db.execute("SELECT title FROM todos WHERE instr(title, ?) > 0 ORDER BY id", ("%",)).fetchall()
print(matches)

[('복습 50%',)]


복습 50% 한 건만 나옵니다. LIKE 조건의 %는 임의 길이의 문자열을 뜻하지만 instr의 두 번째 인자에서는 글자 하나입니다. 입력 문자열은 SQL 뒤에 붙이지 않고 별도 인자로 전달합니다.


### instr에서 %를 한 글자로 찾기

<p>instr(title, ?)에 "%"를 보내면 제목 안에서 %가 처음 나타나는 위치를 돌려줍니다. 없는 경우는 0이므로 조건을 &gt; 0으로 두면 복습 50%만 남습니다.</p>


### 과제 04 · 완료 상태와 검색 조건 결합

완료된 항목 중 제목에 ‘복습’이 들어가는 행을 찾는 조회 함수를 작성하세요. 같은 검색어에서 완료 상태만 바꾸면 다른 행이 나와야 합니다.


In [4]:
def find_titles(completed, term):
    # 검색어나 완료 값을 SQL 문자열에 직접 붙이지 말고 매개변수로 전달하세요.
    pass

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def find_titles(completed, term):
    query = &quot;SELECT title FROM todos WHERE completed=? AND instr(title,?)&gt;0 ORDER BY id&quot;
    return [row[0] for row in db.execute(query, (completed, term))]

print(find_titles(True, &quot;복습&quot;))
print(find_titles(False, &quot;복습&quot;))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;복습 마무리&#x27;]
[&#x27;복습 50%&#x27;]</code></pre>
<p>두 제목 모두 ‘복습’을 포함합니다. 두 결과가 나뉘는 이유는 completed 조건입니다. 검색어에 %를 넣어도 instr는 와일드카드로 확장하지 않고 실제 % 문자가 있는지 찾습니다.</p>
</details>


## 돌아보기

검색 결과가 비었다면 같은 DB에서 필터 없는 SELECT로 먼저 행을 읽으세요. 행이 있으면 검색 조건을, 테이블이 없으면 열었던 파일의 경로를 확인할 수 있습니다.
